# Warm-up 4A · Community Rules: Comment Moderation · Kiểm duyệt bình luận theo nội quy nhóm (NLP · Xử lý ngôn ngữ tự nhiên)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Each row is one comment posted in a Vietnamese online community together with **one rule** of that community. Predict the probability that the comment **breaks that rule** (`rule_violation`). The same comment can break one rule and be perfectly fine under another. | Mỗi dòng là một bình luận trong một nhóm cộng đồng mạng, kèm **một nội quy** của nhóm. Dự đoán xác suất bình luận **vi phạm đúng nội quy đó** (`rule_violation`). Cùng một bình luận có thể vi phạm nội quy này nhưng hoàn toàn hợp lệ với nội quy khác. |
| **Data** | `train/` 8,000 comments (with labels) checked against 8 rules · `public_test/` 3,000 · `private_test/` 3,000. Every row also shows 2 comments that **broke** the rule and 2 that **did not**. Some rules in the test sets **never appear in `train/`** (more of them in private). | `train/` 8.000 bình luận (có nhãn) với 8 nội quy · `public_test/` 3.000 · `private_test/` 3.000. Mỗi dòng còn kèm 2 bình luận **đã vi phạm** và 2 bình luận **không vi phạm** nội quy đó. Một số nội quy trong tập test **không hề có trong `train/`** (tập private có nhiều hơn). |
| **Metric** | Column-averaged ROC-AUC: one ROC-AUC per rule, then the mean over rules, **higher is better**. Ranking uses the **private** score. | ROC-AUC trung bình theo nội quy: tính ROC-AUC cho từng nội quy rồi lấy trung bình, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,rule_violation` (a score per comment; only the order **within each rule** matters) | `public_submission.csv`, `private_submission.csv` gồm cột `id,rule_violation` (điểm cho mỗi bình luận; chỉ thứ tự **trong cùng một nội quy** là quan trọng) |
| **Rules** | PyTorch model trained from scratch: **no pretrained language models** (no PhoBERT/BERT/LLM weights) and no LLM APIs. Classic tools (e.g. scikit-learn TF-IDF) are fine. No external data, no hand-labelling of test comments. Every column of the test files is an input you may use. | Mô hình PyTorch huấn luyện từ đầu: **không dùng mô hình ngôn ngữ huấn luyện sẵn** (PhoBERT/BERT/LLM) và không gọi API LLM. Được dùng công cụ cổ điển (vd. TF-IDF của scikit-learn). Không dùng dữ liệu ngoài, không tự gán nhãn bình luận test. Mọi cột trong file test đều là đầu vào được phép dùng. |

$$\text{Score}=\frac{1}{|R|}\sum_{r\in R}\text{AUC}_r,\qquad R=\text{the rules present in the split · các nội quy có trong tập}$$

**Columns · Các cột:**
- `id` — comment id · mã bình luận
- `group` — the community where it was posted · nhóm cộng đồng nơi bình luận được đăng
- `rule` — the rule to check, in Vietnamese · nội quy cần kiểm tra
- `body` — the comment: teencode, missing diacritics, emoji and disguised phone numbers are common · nội dung bình luận: thường có teencode, viết không dấu, emoji, số điện thoại viết "lách"
- `positive_example_1`, `positive_example_2` — comments that broke this rule · bình luận đã vi phạm nội quy này
- `negative_example_1`, `negative_example_2` — comments that did not break it · bình luận không vi phạm
- `rule_violation` — target (0/1, decided by moderators, so a little noisy), train only · nhãn (0/1, do người kiểm duyệt quyết định nên có chút nhiễu), chỉ có ở train

🇬🇧 Some comments contain rude language: that is what moderators deal with every day.
🇻🇳 Một số bình luận có lời lẽ thô tục: đó là việc người kiểm duyệt phải xử lý hằng ngày.

Inspired by · Lấy cảm hứng từ: [Jigsaw – Agile Community Rules Classification (Kaggle, 2025)](https://www.kaggle.com/competitions/jigsaw-agile-community-rules).
The data here are synthetic · Dữ liệu ở đây là dữ liệu mô phỏng.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from collections import Counter

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "4A"
FOLDER  = "4A_comment_moderation"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-4"

# Already have the folder? Point this at the parent of 4A_comment_moderation/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 4A_comment_moderation/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/4A_comment_moderation/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
MIN_FREQ = 2
EMB_DIM = 64
HIDDEN = 64
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Tokenisation · Đọc dữ liệu và tách từ

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
public_df = pd.read_csv(PUBLIC_CSV)
private_df = pd.read_csv(PRIVATE_CSV)
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")

TARGET = "rule_violation"
print(f"Violation rate / Tỷ lệ vi phạm: {train_df[TARGET].mean():.3f}")
print("Rules in train / Nội quy trong train:")
print(train_df["rule"].value_counts().to_string())


def tokenize(text):
    """EN: lowercase, then split into words (Vietnamese diacritics are kept).
    VI: chuyển chữ thường rồi tách thành các từ (giữ nguyên dấu tiếng Việt)."""
    return re.findall(r"\w+", str(text).lower())


# EN: vocabulary = words seen at least MIN_FREQ times in the training comments; 0 = unknown word
# VI: từ điển = các từ xuất hiện ít nhất MIN_FREQ lần trong bình luận train; 0 = từ lạ
counts = Counter(w for text in train_df["body"] for w in tokenize(text))
vocab = {w: i + 1 for i, (w, c) in enumerate(counts.most_common()) if c >= MIN_FREQ}
print(f"Vocabulary size / Kích thước từ điển: {len(vocab) + 1}")


def encode(text):
    """EN: comment -> tensor of word ids. VI: bình luận -> tensor chỉ số từ."""
    return torch.tensor([vocab.get(w, 0) for w in tokenize(text)] or [0], dtype=torch.long)


def make_items(df):
    """EN: list of (word ids of the comment, label); test rows get a dummy label 0.
    The baseline reads only `body`: it ignores the rule and the example comments.
    VI: danh sách (chỉ số từ của bình luận, nhãn); dòng test có nhãn giả 0.
    Baseline chỉ đọc `body`: bỏ qua nội quy và các bình luận ví dụ."""
    labels = df[TARGET].values if TARGET in df else np.zeros(len(df))
    return [(encode(t), float(y)) for t, y in zip(df["body"], labels)]


def collate(batch):
    """EN: concatenate the word ids of a batch; `offsets` marks where each comment starts (for nn.EmbeddingBag).
    VI: nối chỉ số từ của cả lô; `offsets` đánh dấu vị trí bắt đầu mỗi bình luận (cho nn.EmbeddingBag)."""
    ids = [x for x, _ in batch]
    offsets = torch.tensor([0] + [len(x) for x in ids[:-1]]).cumsum(0)
    return torch.cat(ids), offsets, torch.tensor([y for _, y in batch], dtype=torch.float32)


# EN: random 10% validation split / VI: tách ngẫu nhiên 10% làm tập validation
tr_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df[TARGET])
train_loader = DataLoader(make_items(tr_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(make_items(val_df), batch_size=512, shuffle=False, collate_fn=collate)
public_loader = DataLoader(make_items(public_df), batch_size=512, shuffle=False, collate_fn=collate)
private_loader = DataLoader(make_items(private_df), batch_size=512, shuffle=False, collate_fn=collate)
print(f"Train: {len(tr_df)} | Validation: {len(val_df)}")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class BagOfWords(nn.Module):
    """EN: average of the word embeddings of the comment -> small MLP -> 1 logit.
    VI: trung bình embedding các từ của bình luận -> MLP nhỏ -> 1 logit."""

    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden=HIDDEN):
        super().__init__()
        self.embed = nn.EmbeddingBag(vocab_size, emb_dim, mode="mean")
        self.head = nn.Sequential(nn.Linear(emb_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, ids, offsets):
        return self.head(self.embed(ids, offsets)).squeeze(-1)


model = BagOfWords(len(vocab) + 1).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict_proba(model, loader):
    """EN: probability of a violation for every row. / VI: xác suất vi phạm cho từng dòng."""
    model.eval()
    out = []
    for ids, offsets, _ in loader:
        out.append(torch.sigmoid(model(ids.to(DEVICE), offsets.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


def column_averaged_auc(df, scores):
    """EN: the competition metric: ROC-AUC inside each rule, then the mean over rules.
    VI: độ đo của cuộc thi: ROC-AUC trong từng nội quy, rồi lấy trung bình."""
    d = pd.DataFrame({"rule": df["rule"].values, "y": df[TARGET].values, "p": scores})
    return float(np.mean([roc_auc_score(g["y"], g["p"]) for _, g in d.groupby("rule") if g["y"].nunique() == 2]))


criterion = nn.BCEWithLogitsLoss()  # model outputs logits / mô hình trả về logit
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_auc = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for ids, offsets, y in train_loader:
        ids, offsets, y = ids.to(DEVICE), offsets.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(ids, offsets), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)

    val_auc = column_averaged_auc(val_df, predict_proba(model, val_loader))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(tr_df):.4f} | Val column-avg AUC={val_auc:.4f}"
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation column-averaged AUC: {best_auc:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, scores, csv_path):
    """Save <id, rule_violation> CSV and zip it. / Lưu dự đoán ra CSV và nén zip."""
    pd.DataFrame({"id": ids, TARGET: scores}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict_proba(model, public_loader), "public_submission.csv")
export(private_df["id"], predict_proba(model, private_loader), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", TARGET], f"{csv_path}: columns must be ['id', '{TARGET}']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGET].notna().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert np.isfinite(sub[TARGET]).all(), f"{csv_path}: scores must be finite numbers / điểm phải là số hữu hạn"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "4A"          # which problem this notebook is for
CONTEST = "warmup-4"    # this notebook is for Warmup Contest 4
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
